# Clash Royale — Scrape Card Stats (Fandom)

The official API only gives id/name/elixir/rarity. This notebook scrapes per-card
attributes from the Fandom *Cards* page and writes **`card_attributes.csv`**, keyed by
the API card name so it joins cleanly to `cards.csv`.

It combines every card table that carries real data:
* **combat stats** (troops/buildings/spells/champions): hitpoints, damage, dps,
  attack_period, range, radius, lifetime, crown_tower_damage, special_damage
* **spawn info** (spawner cards): troop_spawned, spawn_count_period, max_troops_spawned
* **evolution info**: evo_cycles, evo_overall_cost, evo_stat_boosts

Pipeline: robust fetch (browser UA + retry + cache) -> parse tables -> keep tables
with a `Card` column -> clean/typed columns -> normalize wiki names to API spelling
-> merge -> save + report.

In [ ]:
import io, os, re, sys, time
import requests
import pandas as pd

sys.path.insert(0, os.getcwd())
import config  # for config.CARDS_CSV

URL = "https://clashroyale.fandom.com/wiki/Cards"
HEADERS = {
    "User-Agent": ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                   "(KHTML, like Gecko) Chrome/120.0 Safari/537.36"),
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
}
CACHE = "scrape_cache.html"
REFRESH = False  # set True to force a fresh download

In [ ]:
# Robust fetch: Fandom intermittently returns 403, so retry. Cache to avoid re-hammering.
if os.path.exists(CACHE) and not REFRESH:
    html = open(CACHE, encoding="utf-8").read()
    print(f"Loaded cached {CACHE} ({len(html)} bytes). Set REFRESH=True to re-download.")
else:
    html = None
    for attempt in range(1, 6):
        resp = requests.get(URL, headers=HEADERS, timeout=30)
        print(f"attempt {attempt}: HTTP {resp.status_code}")
        if resp.status_code == 200:
            html = resp.text
            open(CACHE, "w", encoding="utf-8").write(html)
            break
        time.sleep(1.5)
    if html is None:
        raise SystemExit("Could not fetch the page (Fandom kept blocking). Re-run shortly.")

In [ ]:
# Parse every <table>. NOTE: pandas 3.x requires a file-like object, hence io.StringIO.
tables = pd.read_html(io.StringIO(html))
print(f"Found {len(tables)} tables. Shapes:")
for i, t in enumerate(tables):
    has_card = "Card" in [str(c) for c in t.columns]
    print(f"  {i+1:2d}: {str(t.shape):10s} card_table={has_card}")

In [ ]:
# Column maps. Numeric combat stats -> CANONICAL; descriptive spawn/evo data -> EXTRA.
CANONICAL = ["hitpoints", "damage", "damage_per_second", "attack_period",
             "range", "radius", "lifetime", "crown_tower_damage", "special_damage"]
EXTRA_COLS = ["troop_spawned", "spawn_count_period", "max_troops_spawned",
              "evo_cycles", "evo_overall_cost", "evo_stat_boosts"]


def clean_number(val):
    '''"366 (122x3)" -> 366.0 ; "1,356" -> 1356.0 ; "Upon breaking" -> NaN'''
    m = re.search(r"\d[\d,]*\.?\d*", str(val))
    return float(m.group(0).replace(",", "")) if m else float("nan")


def canon_col(header):
    h = str(header).lower()
    if "damage per second" in h or h.strip() == "dps":
        return "damage_per_second"
    if "crown tower damage" in h:
        return "crown_tower_damage"
    if "special damage" in h:
        return "special_damage"
    if "hitpoint" in h or "health" in h:
        return "hitpoints"
    if "attack period" in h:
        return "attack_period"
    if "damage" in h:
        return "damage"
    if "range" in h:
        return "range"
    if "radius" in h:
        return "radius"
    if "lifetime" in h:
        return "lifetime"
    return None


# Descriptive extras: (substring of lowercased header) -> (output column, is_numeric).
# Order matters: more specific keys first.
_EXTRA = [
    ("maximum troops", ("max_troops_spawned", False)),
    ("spawn count",    ("spawn_count_period", False)),
    ("troop spawned",  ("troop_spawned", False)),
    ("cycles",         ("evo_cycles", True)),
    ("overall cost",   ("evo_overall_cost", True)),
    ("stat boost",     ("evo_stat_boosts", False)),
]


def extra_col(header):
    h = str(header).lower()
    for key, out in _EXTRA:
        if key in h:
            return out
    return None


def clean_text(series):
    s = series.astype(str).str.strip()
    return s.replace({"nan": pd.NA, "None": pd.NA, "": pd.NA})

In [ ]:
# Keep tables with a 'Card' column; pull numeric combat stats AND descriptive extras.
frames = []
for t in tables:
    t.columns = [str(c) for c in t.columns]
    if "Card" not in t.columns:
        continue
    sub = pd.DataFrame({"card": t["Card"].astype(str).str.strip()})
    for col in t.columns:
        canon = canon_col(col)
        if canon and canon not in sub.columns:
            sub[canon] = t[col].map(clean_number)
            continue
        ex = extra_col(col)
        if ex and ex[0] not in sub.columns:
            name, is_num = ex
            sub[name] = t[col].map(clean_number) if is_num else clean_text(t[col])
    sub = sub[sub["card"].notna() & ~sub["card"].isin(["nan", "Card"])]
    if sub.shape[1] > 1:  # has at least one data column besides 'card'
        frames.append(sub)

big = pd.concat(frames, ignore_index=True)
# A card can appear in several tables; .first() coalesces the first non-null per column.
merged = big.groupby("card", as_index=False).first()
print(f"Merged {len(frames)} tables -> {len(merged)} unique cards")
print("  combat stats:", [c for c in CANONICAL if c in merged.columns])
print("  extra cols:  ", [c for c in EXTRA_COLS if c in merged.columns])

In [ ]:
# Normalize wiki names to API spelling so the result joins to cards.csv.
api_names = set(pd.read_csv(config.CARDS_CSV)["name"])
ALIASES = {}  # add wiki->api fixes here if the report below shows real mismatches


def to_api_name(wiki):
    n = str(wiki).strip()
    if n in api_names:
        return n
    if n.rstrip(".") in api_names:   # handles "P.E.K.K.A." / "Mini P.E.K.K.A."
        return n.rstrip(".")
    return ALIASES.get(n)


merged["name"] = merged["card"].map(to_api_name)
matched = merged[merged["name"].notna()].copy()

ORDER = CANONICAL + EXTRA_COLS
out = matched[["name"] + [c for c in ORDER if c in matched.columns]]
out = out.sort_values("name").reset_index(drop=True)
out.to_csv("card_attributes.csv", index=False)

unmatched_wiki = sorted(merged.loc[merged["name"].isna(), "card"].unique())
api_no_stats = sorted(api_names - set(matched["name"]))
print(f"Wrote card_attributes.csv: {len(out)} cards x {out.shape[1]} cols")
print(f"  columns: {list(out.columns)}")
print(f"\nUnmatched wiki rows ({len(unmatched_wiki)}) "
      f"-- expect tower troops / spawned sub-units / removed / Merge Tactics:\n  {unmatched_wiki}")
print(f"\nAPI cards with NO scraped data ({len(api_no_stats)}):\n  {api_no_stats}")

### Merge the attributes into cards.csv
This adds every attribute column onto `cards.csv`, keyed by `name`. It's idempotent
(drops any previously-merged attribute columns first), so you can re-run it safely.

Run order for a full rebuild: **fetch_cards.ipynb** (writes base `cards.csv` from the
API) -> **this notebook** (enriches it). Re-running fetch_cards rebuilds the 7 base
columns, so re-run this notebook afterwards to re-add the attributes.

In [ ]:
attrs = pd.read_csv("card_attributes.csv")
attr_cols = [c for c in attrs.columns if c != "name"]

base = pd.read_csv(config.CARDS_CSV)
base = base.drop(columns=[c for c in attr_cols if c in base.columns])  # idempotent
combined = base.merge(attrs, on="name", how="left")
combined.to_csv(config.CARDS_CSV, index=False)
print(f"Updated {config.CARDS_CSV}: {combined.shape[0]} cards x {combined.shape[1]} cols")
print("columns:", list(combined.columns))
combined.head(20)